# 02 — Feature engineering: *From PGN to 77 features*

Notebook 01 showed that a single game *contains* rating signal. This notebook shows **exactly how we
extract it** — how a raw PGN movetext (SAN moves plus stored `[%eval]` and `[%clk]` annotations)
becomes the 77-number feature vector the model consumes. It is the bridge between the data and the
modelling (rubric: EDA / method bridge).

**Three things to take away:**

1. **The move-quality features rest on *verified* primitives.** Win%, per-move Accuracy%, and move
   classification are transcribed verbatim from Lichess's own source (`lila` / `scalachess`) and
   pinned by unit tests. We are not inventing a quality metric; we reproduce Lichess's.
2. **Absence is encoded as information, not as zero.** The missing-value contract — `NaN` means
   *unobservable*, `0` is reserved for a true zero or an exposure count, and explicit `has_*` flags —
   is what lets a tree treat "this game never reached an endgame" as a *fact about the game* rather
   than as "zero endgame errors." **37.5% of games have no endgame at all.**
3. **The opponent never enters a feature.** Every feature describes one player's own play; the
   leakage guard from notebook 01 (§8) is enforced here by construction.

We will build one real game up by hand — move by move — using the same `src.features` functions the
pipeline uses, then confirm our hand computation matches `extract_player_features` exactly.

In [ ]:
# --- Setup ----------------------------------------------------------------------------------
import sys, io, warnings
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import chess, chess.pgn
from IPython.display import Image, display

from src.config import load_config
from src.plotting import set_style, save_fig
# The verified pure primitives + the phase-split extractor and its internal helpers. We reuse the
# internal helpers (_white_cp, _parse_time_control) so the by-hand worked example is faithful to what
# extract_player_features does inside — not a re-implementation.
from src.features import (
    win_percent, winning_chances, accuracy_percent, classify_move, game_phase,
    extract_player_features, _white_cp, _parse_time_control,
    BLUNDER_THRESHOLD, MISTAKE_THRESHOLD, INACCURACY_THRESHOLD,
)

warnings.filterwarnings("ignore")
cfg = load_config()
set_style()
PROC = Path(cfg["paths"]["processed"])
FIG = Path(cfg["paths"]["figures"]) / "notebook_02_features"   # this notebook's figure folder
CLAMP = cfg["eval"]["cp_clamp"]         # 1000 — evals clamped to +/-1000 before Win%
START_CP = cfg["eval"]["start_cp"]      # 15 — assumed eval of the start position

def show(path):
    display(Image(filename=str(path)))

feats = pd.read_parquet(PROC / "features_300k_v2.parquet")
print("features:", feats.shape, "| clamp", CLAMP, "| start_cp", START_CP)

## 1. The verified primitives

Two different scales are in play, and the code is careful never to conflate them:

- **Win%** ∈ [0, 100] and **winningChances** ∈ [−1, +1] come from the *same* logistic map of the
  centipawn eval, `2 / (1 + exp(-0.00368208·cp)) − 1`, with the constant `MULTIPLIER = -0.00368208`
  copied from Lichess. Win% is just `50 + 50·winningChances`. Accuracy% is computed from the **Win%
  points** lost by a move (`winDiff = win_before − win_after`) via Lichess's
  `103.1668·exp(−0.04354·winDiff) − 3.1669 + 1`.
- **Move classification** (inaccuracy / mistake / blunder) thresholds the drop in **winningChances**
  (the −1..+1 scale) at **0.10 / 0.20 / 0.30** — *not* raw centipawns. A blunder is a move that
  throws ≥ 0.30 of winning chances (≈ 15 Win% points).

Below we plot both curves and print the classification thresholds, then tabulate a few evals to make
the mapping concrete.

In [ ]:
cp = np.linspace(-1000, 1000, 400)
wpct = np.array([win_percent(c, CLAMP) for c in cp])
wdiff = np.linspace(0, 40, 200)                       # Win% points lost by a move
acc = np.array([accuracy_percent(60.0, 60.0 - d) for d in wdiff])   # accuracy depends only on the drop

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.plot(cp, wpct, color="#2c6fbb")
ax1.axhline(50, color="k", ls="--", lw=1); ax1.axvline(0, color="k", ls=":", lw=1)
ax1.set(xlabel="Centipawns (white POV)", ylabel="Win%", title="Win% = logistic(cp)  [Lichess]")
ax2.plot(wdiff, acc, color="#c0504d")
for thr, lab in [(5, "inaccuracy~"), (10, "mistake~"), (15, "blunder~")]:
    ax2.axvline(thr, color="0.5", ls=":", lw=1)
ax2.set(xlabel="Win% points lost by the move", ylabel="Move Accuracy%",
        title="Accuracy% decays with Win% lost")
fig.tight_layout()
show(save_fig(fig, "feat_primitives", FIG))

print(f"classification thresholds on winningChances drop (-1..+1 scale):")
print(f"  inaccuracy >= {INACCURACY_THRESHOLD}   mistake >= {MISTAKE_THRESHOLD}   blunder >= {BLUNDER_THRESHOLD}")
demo = pd.DataFrame({"cp": [-800, -400, -200, -100, 0, 100, 200, 400, 800]})
demo["win_percent"] = demo["cp"].map(lambda c: round(win_percent(c, CLAMP), 1))
demo["winning_chances"] = demo["cp"].map(lambda c: round(winning_chances(c, CLAMP), 3))
display(demo)

**Reading it.** The Win% curve is steepest near equality (a 100 cp swing around 0 moves Win% far
more than a 100 cp swing when already ±600), which is why chess accuracy metrics are defined in Win%
space, not raw centipawns — losing 100 cp from a dead-drawn position is a real error, losing 100 cp
when already up a queen is noise. Accuracy% then decays smoothly from 100 as a move sheds Win%. The
classification thresholds are deliberately on the winningChances scale, so "blunder" means the same
thing (throwing away real winning chances) whether the position is sharp or quiet.

## 2. A worked example: one game, move by move

We pick one real game deterministically — a ~1900-rated player's game that has an opening, a
middlegame, an endgame, and a clock scramble, plus a couple of blunders to look at — and replay it
exactly the way `extract_player_features` does: push each move, read the eval annotated *after* it,
flip to the mover's point of view, and score the move.

In [ ]:
# Deterministic pick: a white-side instance with all three phases, a scramble, and >=1 blunder.
cand = feats[(feats.color == "white") & (feats.has_endgame == 1) & (feats.has_scramble == 1)
             & feats.game_plies.between(60, 90) & (feats.blunder_count >= 1)].sort_values("game_id")
ex = cand.iloc[0]
GID, COLOR = ex["game_id"], ex["color"]

meta_cols = ["game_id", "white", "black", "white_elo", "black_elo", "result", "eco",
             "opening", "time_control", "movetext"]
gc = pd.read_parquet(PROC / "games_clean_300k.parquet", columns=meta_cols)
g = gc[gc.game_id == GID].iloc[0]

game = chess.pgn.read_game(io.StringIO(g["movetext"]))
base, inc = _parse_time_control(g["time_control"])
print(f"game     : {GID}")
print(f"players  : {g['white']} ({g['white_elo']}) vs {g['black']} ({g['black_elo']})  ->  {g['result']}")
print(f"opening  : {g['eco']}  {g['opening']}")
print(f"control  : {g['time_control']}  (base={base}s inc={inc}s)   we describe: {COLOR}  (rating {ex['rating']})")

In [ ]:
# Replay for our player, using the SAME primitives + phase logic as src.features. The eval annotated
# on a move is the eval of the position AFTER it (white POV); win_before is the previous ply's eval.
want_white = (COLOR == "white")
board = game.board()
prev_white_cp = START_CP
rows = []
for node in game.mainline():
    move = node.move
    mover_white = board.turn                 # side to move BEFORE the push == the mover
    san = board.san(move)
    board.push(move)
    after_w = _white_cp(node.eval(), CLAMP)  # white-POV cp after the move (mate folded, clamped)
    if mover_white == want_white and after_w is not None and prev_white_cp is not None:
        mover_after = after_w if want_white else -after_w
        mover_before = prev_white_cp if want_white else -prev_white_cp
        wb, wa = win_percent(mover_before, CLAMP), win_percent(mover_after, CLAMP)
        rows.append({
            "ply": node.ply(),
            "phase": game_phase(node.ply(), board, cfg),
            "move": san,
            "cp_after": mover_after,
            "win_before": round(wb, 1),
            "win_after": round(wa, 1),
            "cpl": max(0.0, mover_before - mover_after),
            "accuracy": round(accuracy_percent(wb, wa), 1),
            "class": classify_move(winning_chances(mover_before, CLAMP), winning_chances(mover_after, CLAMP)),
        })
    prev_white_cp = after_w

movetable = pd.DataFrame(rows)
print(f"{len(movetable)} scored moves for {COLOR}. Worst 6 by centipawn loss:")
display(movetable.sort_values("cpl", ascending=False).head(6))
print("\nFirst 10 scored moves:")
display(movetable.head(10))

**Reading the move table.** Each row is one of our player's moves: the eval after it (mover POV), the
Win% before/after, the centipawn loss (`cpl`, floored at 0 — a move that *improves* the eval is not
penalised), the Lichess Accuracy% for that move, and its class. The worst moves — the blunders — are
exactly the ones that shed a large chunk of Win% in a single move. This per-move record is what the
aggregates summarise: mean/median/max CPL, mean Accuracy%, and the counts/rates of
inaccuracies/mistakes/blunders, computed overall *and separately for each phase*.

In [ ]:
# Cross-check: our by-hand aggregates must match extract_player_features exactly.
fv = extract_player_features(game, want_white, cfg, increment=inc, base_seconds=base,
                             player_result=ex["result"])
manual_cpl_mean = movetable["cpl"].mean()
manual_blunders = int((movetable["class"] == "blunder").sum())
print(f"cpl_mean   : by-hand {manual_cpl_mean:.4f}   |   extract_player_features {fv['cpl_mean']:.4f}")
print(f"blunders   : by-hand {manual_blunders}         |   extract_player_features {int(fv['blunder_count'])}")
print(f"n_moves    : by-hand {len(movetable)}          |   extract_player_features {int(fv['n_moves'])}")
assert abs(manual_cpl_mean - fv["cpl_mean"]) < 1e-6 and manual_blunders == int(fv["blunder_count"])
print("\nMATCH — the notebook replay reproduces src.features exactly.")

In [ ]:
# The full feature vector this one game produces, grouped by phase (a compact view of the 77 numbers).
def phase_block(prefix):
    keys = [f"{prefix}n_moves", f"{prefix}cpl_mean", f"{prefix}cpl_max", f"{prefix}acc_mean",
            f"{prefix}blunder_count", f"{prefix}blunder_rate"]
    return {k.replace(prefix, ""): fv.get(k) for k in keys}

blocks = pd.DataFrame({
    "overall":    phase_block(""),
    "opening":    phase_block("opening_"),
    "middlegame": phase_block("middlegame_"),
    "endgame":    phase_block("endgame_"),
}).T
display(blocks.round(2))
print("presence flags:", {k: fv[k] for k in ["has_opening", "has_middlegame", "has_endgame", "has_clock", "has_scramble"]})

## 3. Phase splitting

Move quality is aggregated per phase because — as notebook 01 showed — the opening separates players
far more than the endgame does. The boundaries are pragmatic (`config.yaml → phases`):

- **Opening**: up to ply `opening_max_ply = 30` (~15 full moves) — a proxy for "still in book."
- **Endgame**: once the total non-pawn, non-king material on the board falls to
  `endgame_nonpawn_pieces = 6` or fewer.
- **Middlegame**: everything in between.

Here is our example game's eval trajectory, coloured by the phase each ply falls in.

In [ ]:
# White-POV eval over the whole game, with phase shading, plus our player's blunders marked.
board = game.board()
plies, cps, phases = [], [], []
for node in game.mainline():
    board.push(node.move)
    w = _white_cp(node.eval(), CLAMP)
    if w is not None:
        plies.append(node.ply()); cps.append(w); phases.append(game_phase(node.ply(), board, cfg))

plies, cps = np.array(plies), np.array(cps)
colors = {"opening": "#2c6fbb", "middlegame": "#e07b39", "endgame": "#8064a2"}
fig, ax = plt.subplots(figsize=(10, 4.4))
ax.axhline(0, color="k", lw=1)
for ph, col in colors.items():
    m = np.array([p == ph for p in phases])
    if m.any():
        ax.scatter(plies[m], cps[m], s=14, color=col, label=ph)
# mark our player's blunders on the trajectory
bl = movetable[movetable["class"] == "blunder"]
for _, br in bl.iterrows():
    ax.axvline(br["ply"], color="#c0504d", ls="--", lw=1, alpha=0.7)
ax.set(xlabel="Ply", ylabel="Eval (centipawns, white POV, clamped)",
       title=f"Eval trajectory by phase — {GID.split('/')[-1]}  (red dashes = our player's blunders)")
ax.legend(title="phase")
show(save_fig(fig, "feat_eval_trajectory", FIG))

**Reading it.** The eval walks through the three phases; our player's blunders (red) are the sharp
vertical swings away from their favour. Because the model gets the *per-phase* CPL and blunder rates
separately, it can learn that (say) a clean opening followed by an endgame collapse is a different
strength signature than the reverse — information that a single overall CPL would blur out.

## 4. The missing-value contract

This is the single most important design decision in the feature stage. The rule:

> **`NaN` means *not observable*. `0` is reserved for a true zero — an exposure count
> (`{phase}_n_moves`, `n_timed_moves`) or a genuinely computed zero. A phase that never happened gets
> `n_moves = 0` and `NaN` for *every* other aggregate in that block, including the error counts —
> because a rate or a count is only meaningful conditional on exposure.** Five presence flags
> (`has_opening`, `has_middlegame`, `has_endgame`, `has_clock`, `has_scramble`) make absence directly
> splittable.

The alternative — writing `0` for "no endgame blunders" when there was no endgame — would tell the
model a lie: "this player played a flawless endgame," when in fact they played none. LightGBM
consumes `NaN` natively (it learns a default split direction), so we never impute or use sentinels.

In [ ]:
# Phase/clock presence across the whole dataset, and a direct check of the contract.
flags = ["has_opening", "has_middlegame", "has_endgame", "has_clock", "has_scramble"]
rates = feats[flags].mean()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(flags, rates.values * 100, color="#3aa66f")
for b, v in zip(bars, rates.values):
    ax.text(b.get_x() + b.get_width() / 2, v * 100, f"{v*100:.0f}%", ha="center", va="bottom")
ax.set(ylabel="% of instances present", title="Phase / clock presence", ylim=(0, 108))
ax.tick_params(axis="x", rotation=15)
show(save_fig(fig, "feat_presence_flags", FIG))

# Contract check: when the endgame block is absent, every non-count endgame aggregate is NaN.
no_eg = feats[feats["endgame_n_moves"] == 0]
print(f"instances with no endgame : {len(no_eg):,}  ({len(no_eg)/len(feats)*100:.1f}%)")
print(f"  of those, endgame_cpl_mean is NaN     : {no_eg['endgame_cpl_mean'].isna().mean()*100:.1f}%")
print(f"  of those, endgame_blunder_count is NaN: {no_eg['endgame_blunder_count'].isna().mean()*100:.1f}%")
print(f"  has_endgame flag for those rows       : {no_eg['has_endgame'].mean():.0f} (all zero)")

In [ ]:
# A few rows side by side: presence flag, exposure count (a true 0/int), and one aggregate (NaN vs value).
mix = pd.concat([
    feats[feats["has_endgame"] == 1].head(3),
    feats[feats["has_endgame"] == 0].head(3),
])[["game_id", "has_endgame", "endgame_n_moves", "endgame_cpl_mean", "endgame_blunder_count"]]
mix["game_id"] = mix["game_id"].str.split("/").str[-1]
display(mix.reset_index(drop=True))

**Reading it.** Every game has an opening and a clock; ~92% reach a middlegame; **only 62.5% reach an
endgame**, so 37.5% carry `NaN` throughout their endgame block (not `0`). The mixed table makes the
contract literal: `has_endgame` and `endgame_n_moves` are true `0`s (defined, splittable), while
`endgame_cpl_mean` and `endgame_blunder_count` are `NaN` (undefined, "no evidence").

This encoding was **A/B tested** against the older mixed 0/NaN scheme on identical grouped splits (see
`reports/results.md → missing-value encoding A/B`): the two are *performance-neutral* (ΔMAE ≈ +0.5,
within split noise), and the `NaN`-contract version was adopted **for correctness**, not for a metric
gain — "no endgame existed" is no longer silently encoded as "0 endgame errors," and absence is
explicitly splittable via the `has_*` flags.

## 5. Clock features and the scramble block

Move times are reconstructed from consecutive `[%clk]` readings (`spent = prev_remaining − remaining +
increment`). On top of the ordinary clock features (`move_time_*`, `fast_move_share`,
`time_trouble_share`), the feature stage isolates a **clock-scramble block**: moves played while the
player's remaining clock — *before* the move — was under `scramble_seconds = 30`. The headline
scramble feature is `scramble_cpl_delta = scramble_cpl_mean − cpl_mean`: *how much worse you play once
the flag is looming, relative to your own baseline.*

In [ ]:
sc = feats[feats["has_scramble"] == 1]
print(f"instances with a scramble phase (<30s reached): {len(sc):,}  ({len(sc)/len(feats)*100:.1f}%)")
print(f"scramble_cpl_delta  median {sc['scramble_cpl_delta'].median():+.1f} cpl   mean {sc['scramble_cpl_delta'].mean():+.1f} cpl")

d = sc["scramble_cpl_delta"].clip(-60, 120)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.hist(d, bins=60, color="#4bacc6")
ax.axvline(0, color="k", ls="--", lw=1)
ax.axvline(sc["scramble_cpl_delta"].median(), color="#c0504d", lw=2,
           label=f"median {sc['scramble_cpl_delta'].median():+.1f}")
ax.set(xlabel="scramble_cpl_delta  (scramble CPL − overall CPL)", ylabel="Instances",
       title="Move-quality degradation under time pressure")
ax.legend()
show(save_fig(fig, "feat_scramble_delta", FIG))

**Reading it.** The distribution sits clearly to the right of zero — the median player loses **~+8
centipawns per move** once under 30 seconds, with a long right tail (mean ~+33) of players who fall
apart in the scramble. This is a *real, measurable* behaviour, and it is exactly the kind of signal
one would hope tightens the rating estimate. It is worth flagging now that notebook 05 finds these
scramble features, despite carrying genuine signal, **do not move the tail error** — the existing
clock+quality features already encode what they add. A satisfying feature to build; an instructive
negative result downstream.

## 6. The leakage guard and the final 77 features

Every feature above describes **one player's own play**. Nothing is derived from the opponent — no
opponent rating, no opponent CPL, no rating gap — because (notebook 01, §8) matchmaking makes any
opponent-derived quantity a near-perfect leak of the label. `split_feature_columns` enforces this by
*excluding* a fixed non-feature set and handing the model everything else.

In [ ]:
from src.model import split_feature_columns, _NON_FEATURE, FULL_CAT
from src.evaluate import feature_groups

num, cat = split_feature_columns(feats)
print(f"model features: {len(num)} numeric + {len(cat)} categorical = {len(num) + len(cat)}")
print(f"categorical   : {cat}")
print(f"excluded (never a feature): {sorted(_NON_FEATURE)}")

groups = feature_groups(num + cat)
census = pd.DataFrame(
    [(g, len(cols)) for g, cols in groups.items()], columns=["group", "n_features"]
).sort_values("n_features", ascending=False)
print()
display(census)

**Reading it.** The model sees **77 features** (73 numeric + 4 low-cardinality categoricals:
`result`, `time_control`, `eco`, `color`). Five columns are permanently excluded — `game_id` and
`username` (identifiers), `rating` (the label), `opening` (free-text, superseded by `eco`), and
`opponent_rating` (the leak, reconstructed only for the copy-opponent baseline in notebook 03). The
ablation grouping (`feature_groups`, keyword-based) splits them into **engine** move-quality features
(the large majority), **clock**, **opening** (`eco`), and **style** — the partition notebook 04 uses
to measure what each group is worth.

Grouping is the other half of the guard: because a single player can appear in many games, all
train/validation/calibration/test splits are made **by `username`** (`grouped_train_val_calib_test`),
never by game — otherwise the model could memorise a player and report a fake score.

## Takeaways

- **We reproduce Lichess's own quality metrics**, verified against unit-pinned constants — the
  worked example matched `extract_player_features` to floating-point precision.
- **A game that ends early is itself information.** The missing-value contract (`NaN` = unobservable,
  `0` = true exposure, `has_*` flags) encodes absence honestly; 37.5% of games have no endgame, and
  the model gets to split on that fact.
- **Time pressure is measurable** (median +8 cpl in the scramble) — a genuine signal, though notebook
  05 shows it does not fix the tail.
- **77 features, none of them the opponent's.** The leakage guard is enforced by construction, and
  reinforced by grouped-by-player splits.

**Next:** notebook 03 asks how far *shallow* features get you — the baselines that the full engine
model must beat, including the copy-opponent leak we just spent two notebooks justifying the
exclusion of.